# 模型训练 MT-1（StratifiedKFold）

**特征组：** `IF + hours_since_start + log1p_amount + A_top2`

**协议：** StratifiedKFold + logloss 早停 (`ES_FRAC=0.20`) + 顺序网格调参 + OOF 阈值搜索


## 0. 环境与数据


In [1]:
import warnings
warnings.filterwarnings('ignore')
import json
import numpy as np
import pandas as pd
from itertools import product
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score, log_loss, f1_score,
    precision_score, recall_score, confusion_matrix,
)
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from IPython.display import display

pd.set_option('display.max_columns', 40)
DATA_PATH = '../../input/creditcard.csv'

EARLY_STOPPING_ROUNDS = 50
MAX_BOOST_ROUNDS = 1500
ES_FRAC = 0.20
CV_N_SPLITS = 5
CV_N_SPLITS_TUNE = 3
CV_RANDOM_STATE = 42
TOP_V_K = 2
IF_INPUT_COLS = None
IF_RANDOM_STATE = 42
IF_N_ESTIMATORS = 200
IF_MAX_SAMPLES = 0.5
IF_CONTAMINATION = 'auto'
IF_MAX_NORMAL_SAMPLES = 50_000


def read_creditcard_csv(path: str) -> pd.DataFrame:
    """依次尝试多种编码读取 creditcard.csv。"""
    for kwargs in (
        {'encoding': 'utf-8'},
        {'encoding': 'utf-8', 'encoding_errors': 'replace'},
        {'encoding': 'latin-1'},
    ):
        try:
            return pd.read_csv(path, **kwargs)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError('utf-8', b'', 0, 1, 'failed')


def build_eda_features(data: pd.DataFrame) -> pd.DataFrame:
    """构造 log1p_amount、hours_since_start、is_one_euro 等 EDA 列。"""
    out = data.copy()
    out['log1p_amount'] = np.log1p(out['Amount'])
    out['hours_since_start'] = (out['Time'] // 3600).astype(int)
    out['is_one_euro'] = out['Amount'] == 1.0
    return out


def build_cross_features(data, top_v, gate_col='is_one_euro', prefix='one_euro'):
    """Family A：门控列 × Top-V，仅在门控为真时激活对应 V。"""
    out = data.copy()
    gate = out[gate_col].astype(float)
    new_cols = []
    for v in top_v:
        name = f'{prefix}_{v}'
        out[name] = gate * out[v]
        new_cols.append(name)
    return out, new_cols


def _split_early_stop_set(X_tr, y_tr, es_frac=ES_FRAC, random_state=42):
    """从 train 折再切 20% 分层样本作 logloss 早停监控集。"""
    return train_test_split(X_tr, y_tr, test_size=es_frac, random_state=random_state, stratify=y_tr)


# 与 feature-engineering 4.2 对齐：LGB 用 class_weight，XGB 用 scale_pos_weight
WEIGHT_SCHEMES = {
    'balanced': None,       # LGB balanced / XGB spw
    'spw_sqrt': 'sqrt',     # LGB {1: sqrt(spw)} / XGB sqrt(spw)
    'spw_0.5x': 0.5,
    'spw_2x': 2.0,
    'no_weight': 0.0,       # LGB 无权重 / XGB spw=1
}
WEIGHT_SCHEME_GRID = list(WEIGHT_SCHEMES.keys())


def _apply_weight_scheme(defaults, model_name, spw, weight_scheme='balanced'):
    """按权重方案写入 class_weight 或 scale_pos_weight（镜像 FE make_classifier_spw）。"""
    vk = WEIGHT_SCHEMES[weight_scheme]
    defaults.pop('scale_pos_weight', None)
    defaults.pop('class_weight', None)
    if model_name == 'LightGBM':
        if vk is None:
            defaults['class_weight'] = 'balanced'
        elif vk == 0.0:
            pass
        elif vk == 'sqrt':
            defaults['class_weight'] = {0: 1.0, 1: float(np.sqrt(spw))}
        else:
            defaults['class_weight'] = {0: 1.0, 1: spw * vk}
    else:
        if vk is None:
            defaults['scale_pos_weight'] = spw
        elif vk == 0.0:
            defaults['scale_pos_weight'] = 1.0
        elif vk == 'sqrt':
            defaults['scale_pos_weight'] = float(np.sqrt(spw))
        else:
            defaults['scale_pos_weight'] = spw * vk
    return defaults


def make_classifier(model_name, y_train, params=None):
    """构造 LGB / XGB 分类器；weight_scheme 见 WEIGHT_SCHEMES。"""
    params = dict(params or {})
    spw = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    weight_scheme = params.pop('weight_scheme', 'balanced')
    if weight_scheme not in WEIGHT_SCHEMES:
        raise ValueError(f'未知 weight_scheme: {weight_scheme!r}，可选 {WEIGHT_SCHEME_GRID}')
    if model_name == 'LightGBM':
        defaults = dict(
            n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6, num_leaves=31,
            min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1,
            random_state=42, verbose=-1, n_jobs=-1,
        )
        defaults.update(params)
        defaults = _apply_weight_scheme(defaults, model_name, spw, weight_scheme)
        return lgb.LGBMClassifier(**defaults)
    defaults = dict(
        n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6,
        min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=42, eval_metric='logloss', verbosity=0, n_jobs=-1,
    )
    defaults.update(params)
    defaults['early_stopping_rounds'] = EARLY_STOPPING_ROUNDS
    defaults = _apply_weight_scheme(defaults, model_name, spw, weight_scheme)
    return xgb.XGBClassifier(**defaults)


def fit_classifier(clf, model_name, X_tr, y_tr, X_es=None, y_es=None):
    """训练分类器；提供 eval_set 时按 binary_logloss 早停。"""
    if X_es is None:
        clf.fit(X_tr, y_tr)
        return clf
    if model_name == 'LightGBM':
        clf.fit(
            X_tr, y_tr, eval_set=[(X_es, y_es)], eval_metric='binary_logloss',
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )
    else:
        clf.fit(X_tr, y_tr, eval_set=[(X_es, y_es)], verbose=False)
    return clf


def pick_top_v_features(data, feature_cols, k=TOP_V_K, model_name='LightGBM', random_state=42):
    """在 hold-out 上训练 LGB，按 gain 选 Top-k V 列供 A_top2 交叉。"""
    X, y = data[feature_cols], data['Class']
    X_tr, _, y_tr, _ = train_test_split(X, y, test_size=0.2, random_state=random_state, stratify=y)
    X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_tr, y_tr, random_state=random_state)
    clf = make_classifier(model_name, y_fit)
    fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
    v_cols = [c for c in feature_cols if c.startswith('V')]
    top_v = list(pd.Series(clf.feature_importances_, index=feature_cols)[v_cols].sort_values(ascending=False).head(k).index)
    print(f'Top-{k} V: {top_v}')
    return top_v


def oof_if_anomaly_score(data, feature_cols, y_col='Class', n_splits=CV_N_SPLITS, random_state=IF_RANDOM_STATE, fold_iter=None):
    """OOF Isolation Forest 异常分（越大越异常）；仅用各折 train 正常样本训练。"""
    X = data[feature_cols].values.astype(np.float64)
    y = data[y_col].values
    oof = np.zeros(len(y), dtype=np.float64)
    for fold, (tr_idx, va_idx) in enumerate(fold_iter, start=1):
        normal_tr = tr_idx[y[tr_idx] == 0]
        if len(normal_tr) > IF_MAX_NORMAL_SAMPLES:
            rng = np.random.default_rng(random_state + fold)
            normal_tr = rng.choice(normal_tr, size=IF_MAX_NORMAL_SAMPLES, replace=False)
        scaler = StandardScaler()
        X_n = scaler.fit_transform(X[normal_tr])
        X_va = scaler.transform(X[va_idx])
        iforest = IsolationForest(
            n_estimators=IF_N_ESTIMATORS, max_samples=IF_MAX_SAMPLES,
            contamination=IF_CONTAMINATION, random_state=random_state + fold, n_jobs=-1,
        )
        iforest.fit(X_n)
        oof[va_idx] = -iforest.score_samples(X_va)
        print(f'  IF fold {fold}/{n_splits} 完成 (normal={len(normal_tr):,})')
    return oof


def cross_val_oof(model_name, data, feature_cols, params=None, n_splits=CV_N_SPLITS, random_state=CV_RANDOM_STATE, fold_iter_fn=None):
    """交叉验证：返回 OOF 概率、各折 logloss / AUC-PR。"""
    X, y = data[feature_cols], data['Class']
    oof = np.zeros(len(y))
    fold_ll, fold_ap = [], []
    for tr_idx, va_idx in fold_iter_fn(X, y, n_splits, random_state):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_tr, y_tr, random_state=random_state)
        clf = make_classifier(model_name, y_fit, params=params)
        fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
        proba = clf.predict_proba(X_va)[:, 1]
        fold_ll.append(float(log_loss(y_va, proba, labels=[0, 1])))
        fold_ap.append(float(average_precision_score(y_va, proba)))
        oof[va_idx] = proba
    return oof, fold_ll, fold_ap


def cv_summary(model_name, data, feature_cols, params=None, n_splits=CV_N_SPLITS, random_state=CV_RANDOM_STATE, fold_iter_fn=None, threshold=0.5):
    """汇总 CV 指标（含默认阈值下的分类指标）。"""
    oof, fold_ll, fold_ap = cross_val_oof(model_name, data, feature_cols, params, n_splits, random_state, fold_iter_fn)
    pred = (oof >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(data['Class'], pred).ravel()
    return {
        'model': model_name,
        'logloss_mean': float(np.mean(fold_ll)), 'logloss_std': float(np.std(fold_ll)),
        'AUC-PR_mean': float(np.mean(fold_ap)), 'AUC-PR_std': float(np.std(fold_ap)),
        'F1': f1_score(data['Class'], pred, zero_division=0),
        'Precision': precision_score(data['Class'], pred, zero_division=0),
        'Recall': recall_score(data['Class'], pred, zero_division=0),
        'FP': int(fp), 'FN': int(fn), 'oof_proba': oof,
    }


def sequential_grid_search(model_name, data, feature_cols, param_stages, fold_iter_fn, n_splits=CV_N_SPLITS_TUNE, random_state=CV_RANDOM_STATE):
    """顺序网格：lr×weight → 树结构 → min_child → 采样 → 正则；每阶段用 CV logloss 选优。"""
    best, hist = {}, []

    def score_row(cand):
        p = {**best, **cand}
        _, fold_ll, fold_ap = cross_val_oof(model_name, data, feature_cols, p, n_splits, random_state, fold_iter_fn)
        return {
            **p, 'logloss_mean': float(np.mean(fold_ll)), 'logloss_std': float(np.std(fold_ll)),
            'AUC-PR_mean': float(np.mean(fold_ap)),
        }

    for i, stage in enumerate(param_stages, 1):
        keys = list(stage.keys())
        cands = [dict(zip(keys, v)) for v in product(*stage.values())]
        stage_best = None
        print('')  # 阶段间空行
        print(f'Stage {i} {keys} ({model_name})')
        for cand in cands:
            row = score_row(cand)
            row['stage'] = i
            hist.append(row)
            print(f"  {cand}  logloss={row['logloss_mean']:.5f}  AP={row['AUC-PR_mean']:.4f}")
            if stage_best is None or row['logloss_mean'] < stage_best['logloss_mean'] - 1e-9 or (
                abs(row['logloss_mean'] - stage_best['logloss_mean']) <= 1e-9 and row['AUC-PR_mean'] > stage_best['AUC-PR_mean']
            ):
                stage_best = row
        for k in keys:
            best[k] = stage_best[k]
        print('  >>', {k: best[k] for k in keys})
    return best, pd.DataFrame(hist)


def search_best_threshold(y_true, oof_proba, auc_pr):
    """在 OOF 概率上网格搜索 F1 最优阈值。"""
    thr_grid = np.unique(np.concatenate([
        np.linspace(0.001, 0.05, 50),
        np.linspace(0.06, 0.50, 45),
        np.linspace(0.55, 0.95, 9),
    ]))
    rows = []
    for t in thr_grid:
        pred = (oof_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
        rows.append({
            'threshold': float(t),
            'F1': f1_score(y_true, pred, zero_division=0),
            'Precision': precision_score(y_true, pred, zero_division=0),
            'Recall': recall_score(y_true, pred, zero_division=0),
            'FP': int(fp), 'FN': int(fn), 'AUC-PR': auc_pr,
        })
    scan = pd.DataFrame(rows)
    best = scan.loc[scan['F1'].idxmax()]
    return float(best['threshold']), scan, best


LR_GRID = [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.1]

LGB_STAGES = [
    {'learning_rate': LR_GRID, 'weight_scheme': WEIGHT_SCHEME_GRID},
    {'max_depth': [3, 4, 5, 6, 7, 8], 'num_leaves': [15, 31, 47, 63, 127]},
    {'min_child_samples': [5, 10, 20, 30, 40, 50]},
    {'subsample': [0.6, 0.7, 0.8, 0.9, 1.0], 'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0]},
    {'reg_alpha': [0.0, 0.01, 0.05, 0.1, 0.5, 1.0], 'reg_lambda': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]},
]
XGB_STAGES = [
    {'learning_rate': LR_GRID, 'weight_scheme': WEIGHT_SCHEME_GRID},
    {'max_depth': [3, 4, 5, 6, 7, 8]},
    {'min_child_weight': [1, 2, 3, 5, 7, 10]},
    {'subsample': [0.6, 0.7, 0.8, 0.9, 1.0], 'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0]},
    {'reg_alpha': [0.0, 0.01, 0.05, 0.1, 0.5, 1.0], 'reg_lambda': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]},
]

from sklearn.model_selection import StratifiedKFold

CV_MODE = 'StratifiedKFold'


def make_fold_iter(X, y, n_splits, random_state):
    """生成分层 K 折 (train_idx, val_idx) 迭代器。"""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    yield from skf.split(X, y)


def make_if_fold_iter(data, n_splits, random_state):
    """IF 特征 OOF 用的分层折迭代器。"""
    y = data['Class'].values
    X = np.arange(len(data))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    yield from skf.split(X, y)

df_raw = read_creditcard_csv(DATA_PATH)
V_COLS = [c for c in df_raw.columns if c.startswith('V')]
BASE_FEATURES = V_COLS + ['Amount', 'Time']
IF_INPUT_COLS = V_COLS
print('行数:', f"{len(df_raw):,}", ' 欺诈:', df_raw['Class'].sum(), ' 欺诈率:', f"{df_raw['Class'].mean():.4f}", ' CV:', CV_MODE)


行数: 284,807  欺诈: 492  欺诈率: 0.0017  CV: StratifiedKFold


## 1. 特征工程

| 增量列 | 含义 |
|--------|------|
| `if_oof_score` | OOF Isolation Forest 异常分 |
| `hours_since_start` | `Time // 3600` |
| `log1p_amount` | `log1p(Amount)` |
| `one_euro_V*` | A_top2：`is_one_euro × Top-2 V` |


In [2]:
# --- 构造特征集 IF + hours + log1p + A_top2 ---
df_model = build_eda_features(df_raw)
TOP_V = pick_top_v_features(df_model, BASE_FEATURES, k=TOP_V_K)
df_model, CROSS_A_TOP2 = build_cross_features(df_model, TOP_V)

print('计算 OOF IF 异常分…')
if_fold_iter = list(make_if_fold_iter(df_model, CV_N_SPLITS, CV_RANDOM_STATE))
df_model['if_oof_score'] = oof_if_anomaly_score(
    df_model, IF_INPUT_COLS, n_splits=CV_N_SPLITS, fold_iter=if_fold_iter,
)

EXTRA_FEATURES = ['if_oof_score', 'hours_since_start', 'log1p_amount'] + CROSS_A_TOP2
MODEL_FEATURES = BASE_FEATURES + EXTRA_FEATURES
print('特征数:', len(MODEL_FEATURES))
print('增量列:', EXTRA_FEATURES)


Top-2 V: ['V15', 'V14']
计算 OOF IF 异常分…
  IF fold 1/5 完成 (normal=50,000)
  IF fold 2/5 完成 (normal=50,000)
  IF fold 3/5 完成 (normal=50,000)
  IF fold 4/5 完成 (normal=50,000)
  IF fold 5/5 完成 (normal=50,000)
特征数: 35
增量列: ['if_oof_score', 'hours_since_start', 'log1p_amount', 'one_euro_V15', 'one_euro_V14']


## 2. 基线 CV（默认超参）


In [3]:
baseline_rows = []
for model in ('LightGBM', 'XGBoost'):
    baseline_rows.append(cv_summary(model, df_model, MODEL_FEATURES, fold_iter_fn=make_fold_iter))
baseline_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'oof_proba'} for r in baseline_rows])
display(baseline_df.round(4))


,model,logloss_mean,logloss_std,AUC-PR_mean,AUC-PR_std,F1,Precision,Recall,FP,FN
0,LightGBM,0.1250,0.0684,0.2043,0.2519,0.1514,0.0953,0.3679,1718,311
1,XGBoost,0.0029,0.0005,0.8595,0.0246,0.8635,0.9007,0.8293,45,84


## 3. 顺序网格调参（CV logloss 最小）

顺序：learning_rate×weight_scheme → max_depth/num_leaves → min_child → subsample/colsample → reg

Stage 1 联合搜 `learning_rate`（`LR_GRID`）与 `weight_scheme`（5 种），避免 lr 与不平衡权重交互被顺序锁死。


In [4]:
best_lgb, hist_lgb = sequential_grid_search(
    'LightGBM', df_model, MODEL_FEATURES, LGB_STAGES, make_fold_iter,
)
best_xgb, hist_xgb = sequential_grid_search(
    'XGBoost', df_model, MODEL_FEATURES, XGB_STAGES, make_fold_iter,
)
print('最优 LGB:', best_lgb)
print('最优 XGB:', best_xgb)



Stage 1 ['learning_rate'] (LightGBM)
  {'learning_rate': 0.03}  logloss=0.01679  AP=0.5366
  {'learning_rate': 0.05}  logloss=0.12581  AP=0.0409
  {'learning_rate': 0.07}  logloss=0.12314  AP=0.1830
  >> {'learning_rate': 0.03}

Stage 2 ['max_depth', 'num_leaves'] (LightGBM)
  {'max_depth': 4, 'num_leaves': 15}  logloss=0.02058  AP=0.5635
  {'max_depth': 4, 'num_leaves': 31}  logloss=0.01942  AP=0.5700
  {'max_depth': 4, 'num_leaves': 63}  logloss=0.01942  AP=0.5700
  {'max_depth': 6, 'num_leaves': 15}  logloss=0.01030  AP=0.6863
  {'max_depth': 6, 'num_leaves': 31}  logloss=0.01679  AP=0.5366
  {'max_depth': 6, 'num_leaves': 63}  logloss=0.01540  AP=0.5567
  {'max_depth': 8, 'num_leaves': 15}  logloss=0.00893  AP=0.7310
  {'max_depth': 8, 'num_leaves': 31}  logloss=0.01555  AP=0.5520
  {'max_depth': 8, 'num_leaves': 63}  logloss=0.00940  AP=0.6680
  >> {'max_depth': 8, 'num_leaves': 15}

Stage 3 ['pos_weight_mult'] (LightGBM)
  {'pos_weight_mult': 0.5}  logloss=0.00606  AP=0.7910
  {

## 4. 调参后 5-fold CV + 选模


In [5]:
final_cv = {}
for model, params in [('LightGBM', best_lgb), ('XGBoost', best_xgb)]:
    final_cv[model] = cv_summary(model, df_model, MODEL_FEATURES, params=params, n_splits=CV_N_SPLITS, fold_iter_fn=make_fold_iter)

best_model = max(final_cv, key=lambda m: final_cv[m]['AUC-PR_mean'])
best_params = best_lgb if best_model == 'LightGBM' else best_xgb
oof_proba = final_cv[best_model]['oof_proba']
auc_pr = final_cv[best_model]['AUC-PR_mean']
print('选定模型:', best_model, ' AUC-PR=', f'{auc_pr:.4f}')
display(pd.DataFrame([{k: v for k, v in final_cv[m].items() if k != 'oof_proba'} for m in final_cv]).round(4))


选定模型: XGBoost  AUC-PR= 0.8599


,model,logloss_mean,logloss_std,AUC-PR_mean,AUC-PR_std,F1,Precision,Recall,FP,FN
0,LightGBM,0.0069,0.0043,0.7620,0.1322,0.8216,0.839,0.8049,76,96
1,XGBoost,0.0028,0.0006,0.8599,0.0287,0.8617,0.904,0.8232,43,87


## 5. OOF 阈值搜索 + 最终指标


In [6]:
best_thr, thr_scan, best_row = search_best_threshold(df_model['Class'], oof_proba, auc_pr)
final_metrics = {
    'model': best_model,
    'cv_method': CV_MODE,
    'AUC-PR': auc_pr,
    'best_threshold': best_thr,
    'F1': float(best_row['F1']),
    'Precision': float(best_row['Precision']),
    'Recall': float(best_row['Recall']),
    'FP': int(best_row['FP']),
    'FN': int(best_row['FN']),
}
print('最终指标 @F1 最优阈值:')
for k, v in final_metrics.items():
    print(' ', k + ':', v)
display(thr_scan.sort_values('F1', ascending=False).head(10).round(4))


最终指标 @F1 最优阈值:
  model: XGBoost
  cv_method: StratifiedKFold
  AUC-PR: 0.8599431509184271
  best_threshold: 0.75
  F1: 0.867027027027027
  Precision: 0.9260969976905312
  Recall: 0.8150406504065041
  FP: 32
  FN: 91


,threshold,F1,Precision,Recall,FP,FN,AUC-PR
99,0.75,0.8670,0.9261,0.8150,32,91,0.8599
101,0.85,0.8668,0.9363,0.8069,27,95,0.8599
96,0.60,0.8651,0.9140,0.8211,38,88,0.8599
97,0.65,0.8648,0.9159,0.8191,37,89,0.8599
98,0.70,0.8645,0.9178,0.8171,36,90,0.8599
95,0.55,0.8645,0.9101,0.8232,40,87,0.8599
100,0.80,0.8643,0.9277,0.8089,31,94,0.8599
94,0.50,0.8617,0.9040,0.8232,43,87,0.8599
102,0.90,0.8615,0.9378,0.7967,26,100,0.8599
93,0.49,0.8608,0.9020,0.8232,44,87,0.8599


## 6. 导出


In [7]:
export = {
    'combo': 'IF+hours_since_start+log1p_amount+A_top2',
    'cv_method': CV_MODE,
    'ES_FRAC': ES_FRAC,
    'MODEL_FEATURES': MODEL_FEATURES,
    'best_model': best_model,
    'best_params': best_params,
    'best_lgb_params': best_lgb,
    'best_xgb_params': best_xgb,
    'final_metrics': final_metrics,
    'final_cv': {m: {k: v for k, v in final_cv[m].items() if k != 'oof_proba'} for m in final_cv},
}
with open('model_training_if_log1p_atop2_stratified.json', 'w', encoding='utf-8') as f:
    json.dump(export, f, ensure_ascii=False, indent=2)
print('已导出', 'model_training_if_log1p_atop2_stratified.json')


已导出 model_training_if_log1p_atop2_stratified.json
